# Shopee Malaysia Google Play Reviews Preprocessing

This notebook covers Member 1 responsibilities only: data acquisition, cleaning, NLP preprocessing, basic EDA, dataset validation, and final export.

## 1. Setup and Package Verification

The Shopee Malaysia Google Play package is `com.shopee.my`. It is verified from the Google Play app details URL: `https://play.google.com/store/apps/details?id=com.shopee.my`.

In [ ]:
from collections import Counter
from datetime import datetime
import html
import json
import re
import string
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd
from google_play_scraper import Sort, reviews
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

APP_PACKAGE = "com.shopee.my"
TARGET_REVIEWS = 10_000
BATCH_SIZE = 200
PROGRESS_INTERVAL = 500

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw_data"
CLEAN_DIR = PROJECT_ROOT / "data"
DOCS_DIR = PROJECT_ROOT / "docs"
RAW_PATH = RAW_DIR / "shopee_my_reviews_raw.csv"
CLEAN_PATH = CLEAN_DIR / "cleaned_data.csv"
SUMMARY_PATH = DOCS_DIR / "dataset_summary.md"
KAFKA_SAMPLE_PATH = DOCS_DIR / "kafka_sample_message.json"

for directory in [RAW_DIR, CLEAN_DIR, DOCS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Google Play package name: {APP_PACKAGE}")
print("Verified using: https://play.google.com/store/apps/details?id=com.shopee.my")

## 2. Helper Functions

These functions keep scraping, cleaning, preprocessing, validation, and documentation readable and reusable.

In [ ]:
RAW_COLUMNS = [
    "review_id",
    "rating",
    "review_date",
    "app_version",
    "thumbs_up_count",
    "original_text",
]

FINAL_COLUMNS = RAW_COLUMNS + ["cleaned_text"]

SENTIMENT_WORDS_TO_KEEP = {
    "good",
    "bad",
    "excellent",
    "terrible",
    "love",
    "hate",
    "best",
    "worst",
}

MALAY_STOPWORDS = {
    "ada", "adalah", "akan", "aku", "anda", "apa", "atau", "bagaimana",
    "bagi", "bahawa", "banyak", "baru", "belum", "boleh", "buat", "dalam",
    "dan", "dapat", "dari", "daripada", "dengan", "dia", "di", "ini",
    "itu", "jadi", "jika", "juga", "kami", "kamu", "kan", "ke", "kerana",
    "lagi", "lah", "lebih", "mereka", "nak", "ni", "nya", "oleh", "pada",
    "pun", "saja", "saya", "sebagai", "semua", "sini", "sudah", "tak",
    "telah", "tentang", "tersebut", "tetapi", "tidak", "untuk", "yang",
}


def download_nltk_resources():
    """Download only the NLTK resources required by this notebook."""
    resources = [
        ("corpora/stopwords", "stopwords"),
        ("corpora/wordnet", "wordnet"),
        ("corpora/omw-1.4", "omw-1.4"),
    ]

    for resource_path, package_name in resources:
        try:
            nltk.data.find(resource_path)
        except LookupError:
            nltk.download(package_name, quiet=True)


def scrape_google_play_reviews(app_package, target_reviews):
    """Collect Google Play reviews using continuation tokens until the target or end is reached."""
    collected_reviews = []
    continuation_token = None
    last_reported = 0

    try:
        while len(collected_reviews) < target_reviews:
            remaining = target_reviews - len(collected_reviews)
            batch_count = min(BATCH_SIZE, remaining)

            batch, continuation_token = reviews(
                app_package,
                lang="en",
                country="my",
                sort=Sort.NEWEST,
                count=batch_count,
                continuation_token=continuation_token,
            )

            if not batch:
                print("No more reviews were returned by Google Play.")
                break

            collected_reviews.extend(batch)

            if len(collected_reviews) - last_reported >= PROGRESS_INTERVAL:
                print(f"Collected {len(collected_reviews):,} reviews...")
                last_reported = len(collected_reviews)

            if continuation_token is None:
                print("Reached the final Google Play review page.")
                break
    except Exception as error:
        raise RuntimeError(f"Google Play scraping failed: {error}") from error

    if not collected_reviews:
        raise ValueError("No reviews were collected. Check internet access and package name.")

    print(f"Finished collection with {len(collected_reviews):,} reviews.")
    return collected_reviews


def convert_reviews_to_dataframe(review_records):
    """Keep only project-required fields and standardize column names."""
    rows = []
    for record in review_records:
        rows.append(
            {
                "review_id": record.get("reviewId"),
                "rating": record.get("score"),
                "review_date": record.get("at"),
                "app_version": record.get("reviewCreatedVersion"),
                "thumbs_up_count": record.get("thumbsUpCount"),
                "original_text": record.get("content"),
            }
        )

    dataframe = pd.DataFrame(rows, columns=RAW_COLUMNS)
    dataframe["review_date"] = pd.to_datetime(dataframe["review_date"], errors="coerce")
    dataframe["review_date"] = dataframe["review_date"].dt.strftime("%Y-%m-%d")
    return dataframe


def save_csv(dataframe, path):
    """Save a dataframe to CSV with a clear error if writing fails."""
    try:
        dataframe.to_csv(path, index=False, encoding="utf-8-sig")
    except Exception as error:
        raise OSError(f"Could not save CSV file to {path}: {error}") from error


def contains_letters_or_numbers(text):
    """Return True when a review has readable alphanumeric content."""
    return bool(re.search(r"[A-Za-z0-9]", str(text)))


def has_at_least_three_words(text):
    """Return True when a review has at least three word tokens."""
    return len(re.findall(r"\b\w+\b", str(text))) >= 3


def clean_raw_dataframe(dataframe):
    """Remove duplicates, missing values, empty reviews, symbol-only reviews, and very short reviews."""
    cleaned = dataframe.copy()
    initial_rows = len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.drop_duplicates(subset="review_id")
    duplicate_id_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.drop_duplicates(subset="original_text")
    duplicate_text_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned.dropna(subset=["review_id", "rating", "review_date", "original_text"])
    null_removed = before - len(cleaned)

    cleaned["original_text"] = cleaned["original_text"].astype(str).str.strip()

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"] != ""]
    empty_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"].apply(contains_letters_or_numbers)]
    symbol_only_removed = before - len(cleaned)

    before = len(cleaned)
    cleaned = cleaned[cleaned["original_text"].apply(has_at_least_three_words)]
    short_removed = before - len(cleaned)

    cleaned = cleaned.reset_index(drop=True)

    stats = {
        "initial_rows": initial_rows,
        "duplicate_id_removed": duplicate_id_removed,
        "duplicate_text_removed": duplicate_text_removed,
        "duplicates_removed": duplicate_id_removed + duplicate_text_removed,
        "null_removed": null_removed,
        "empty_removed": empty_removed,
        "symbol_only_removed": symbol_only_removed,
        "short_removed": short_removed,
        "final_cleaned_rows": len(cleaned),
    }

    return cleaned, stats


def remove_emojis(text):
    """Remove common emoji ranges from text."""
    emoji_pattern = re.compile(
        "["
        "\U0001F300-\U0001F5FF"
        "\U0001F600-\U0001F64F"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE,
    )
    return emoji_pattern.sub(" ", str(text))


def get_stopword_set():
    """Create English and Malay stopword set while keeping important sentiment words."""
    english_stopwords = set(stopwords.words("english"))
    combined_stopwords = english_stopwords.union(MALAY_STOPWORDS)
    return combined_stopwords.difference(SENTIMENT_WORDS_TO_KEEP)


def preprocess_text(text, stopword_set, lemmatizer):
    """Apply lowercase, cleaning, tokenization, stopword removal, and English lemmatization."""
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\b[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}\b", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = html.unescape(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", " ", text)
    text = remove_emojis(text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = text.split()
    tokens = [token for token in tokens if token not in stopword_set]
    tokens = [lemmatizer.lemmatize(token, wordnet.VERB) for token in tokens]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(tokens)


def get_top_words(series, top_n=20):
    """Return the most frequent words from a text series."""
    words = []
    for text in series.dropna().astype(str):
        words.extend(re.findall(r"\b[a-zA-Z]{2,}\b", text.lower()))
    return Counter(words).most_common(top_n)


def validate_final_dataset(dataframe):
    """Validate final dataset rules required before export."""
    checks = {
        "review_id contains no duplicates": not dataframe["review_id"].duplicated().any(),
        "cleaned_text contains no null values": dataframe["cleaned_text"].notna().all(),
        "original_text contains no null values": dataframe["original_text"].notna().all(),
        "cleaned_text contains no empty strings": (dataframe["cleaned_text"].astype(str).str.strip() != "").all(),
        "rating only contains values between 1 and 5": dataframe["rating"].between(1, 5).all(),
    }

    for description, passed in checks.items():
        status = "PASS" if passed else "FAIL"
        print(f"{status}: {description}")

    if not all(checks.values()):
        failed = [description for description, passed in checks.items() if not passed]
        raise ValueError(f"Final dataset validation failed: {failed}")


download_nltk_resources()
STOPWORD_SET = get_stopword_set()
LEMMATIZER = WordNetLemmatizer()
print("Helper functions and NLP resources are ready.")

## 3. Data Collection

Reviews are collected with `google-play-scraper`, using continuation tokens until 10,000 reviews are collected or Google Play returns no more reviews.

In [ ]:
review_records = scrape_google_play_reviews(APP_PACKAGE, TARGET_REVIEWS)
raw_df = convert_reviews_to_dataframe(review_records)

if raw_df.empty:
    raise ValueError("Raw dataset is empty after conversion.")

save_csv(raw_df, RAW_PATH)
print(f"Raw dataset saved to: {RAW_PATH}")
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()

## 4. Data Cleaning

This stage removes duplicate review IDs, duplicate review texts, null values, empty reviews, symbol-only reviews, and reviews shorter than three words.

In [ ]:
clean_df, cleaning_stats = clean_raw_dataframe(raw_df)

print(f"Duplicate records removed: {cleaning_stats['duplicates_removed']:,}")
print(f"Null records removed: {cleaning_stats['null_removed']:,}")
print(f"Empty reviews removed: {cleaning_stats['empty_removed']:,}")
print(f"Symbol-only reviews removed: {cleaning_stats['symbol_only_removed']:,}")
print(f"Reviews shorter than three words removed: {cleaning_stats['short_removed']:,}")
print(f"Final number of cleaned reviews before NLP: {cleaning_stats['final_cleaned_rows']:,}")

if clean_df.empty:
    raise ValueError("No reviews remain after cleaning.")

## 5. NLP Preprocessing

The `cleaned_text` column is created by applying lowercase conversion, URL/email/HTML removal, punctuation/number/emoji/special-character removal, whitespace normalization, tokenization, English and Malay stopword removal, and English lemmatization.

In [ ]:
clean_df["cleaned_text"] = clean_df["original_text"].apply(
    lambda text: preprocess_text(text, STOPWORD_SET, LEMMATIZER)
)

before_empty_nlp = len(clean_df)
clean_df = clean_df[clean_df["cleaned_text"].str.strip() != ""].reset_index(drop=True)
empty_after_nlp_removed = before_empty_nlp - len(clean_df)

print(f"Rows removed after NLP because cleaned_text was empty: {empty_after_nlp_removed:,}")
print(f"Final cleaned reviews after NLP: {len(clean_df):,}")
clean_df[["original_text", "cleaned_text"]].head()

## 6. Exploratory Data Analysis

In [ ]:
total_raw_reviews = len(raw_df)
total_cleaned_reviews = len(clean_df)

print("Dataset Summary")
print(f"Total raw reviews collected: {total_raw_reviews:,}")
print(f"Total cleaned reviews: {total_cleaned_reviews:,}")
print(f"Number of duplicates removed: {cleaning_stats['duplicates_removed']:,}")
print(f"Number of missing values removed: {cleaning_stats['null_removed']:,}")
print(f"Dataset dimensions: {clean_df.shape}")

rating_counts = clean_df["rating"].value_counts().sort_index()
rating_percentages = (rating_counts / len(clean_df) * 100).round(2)
average_rating = clean_df["rating"].mean()

print("\nRating Analysis")
print(f"Average rating: {average_rating:.2f}")
print("Rating percentages:")
print(rating_percentages)

plt.figure(figsize=(8, 5))
rating_counts.plot(kind="bar", color="#2F6F6D")
plt.title("Rating Distribution")
plt.xlabel("Rating")
plt.ylabel("Number of Reviews")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

clean_df["original_word_count"] = clean_df["original_text"].apply(lambda text: len(str(text).split()))
clean_df["cleaned_word_count"] = clean_df["cleaned_text"].apply(lambda text: len(str(text).split()))

print("\nReview Analysis")
print(f"Average original review length: {clean_df['original_word_count'].mean():.2f} words")
print(f"Average cleaned review length: {clean_df['cleaned_word_count'].mean():.2f} words")

plt.figure(figsize=(8, 5))
clean_df["original_word_count"].plot(kind="hist", bins=30, color="#8A5A44")
plt.title("Review Length Distribution Before Preprocessing")
plt.xlabel("Word Count")
plt.ylabel("Number of Reviews")
plt.tight_layout()
plt.show()

top_words_before = get_top_words(clean_df["original_text"], 20)
top_words_after = get_top_words(clean_df["cleaned_text"], 20)

print("\nTop 20 words before preprocessing:")
print(top_words_before)
print("\nTop 20 words after preprocessing:")
print(top_words_after)

for title, word_counts, color in [
    ("Top 20 Words Before Preprocessing", top_words_before, "#466A8F"),
    ("Top 20 Words After Preprocessing", top_words_after, "#7A6F36"),
]:
    words = [word for word, count in word_counts]
    counts = [count for word, count in word_counts]
    plt.figure(figsize=(10, 5))
    plt.bar(words, counts, color=color)
    plt.title(title)
    plt.xlabel("Words")
    plt.ylabel("Frequency")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

clean_df = clean_df.drop(columns=["original_word_count", "cleaned_word_count"])

## 7. Validate Final Dataset

In [ ]:
final_df = clean_df[FINAL_COLUMNS].copy()
final_df["rating"] = pd.to_numeric(final_df["rating"], errors="coerce")
final_df["thumbs_up_count"] = pd.to_numeric(final_df["thumbs_up_count"], errors="coerce").fillna(0).astype(int)

validate_final_dataset(final_df)

print("\nDataset shape:")
print(final_df.shape)
print("\nColumn names:")
print(final_df.columns.tolist())
print("\nData types:")
print(final_df.dtypes)
print("\nFirst five records:")
display(final_df.head())

## 8. Export Cleaned Dataset

In [ ]:
save_csv(final_df, CLEAN_PATH)
print(f"Cleaned dataset saved to: {CLEAN_PATH}")
print(f"Final dataset columns: {final_df.columns.tolist()}")

## 9. Dataset Documentation

In [ ]:
collection_date = datetime.now().strftime("%Y-%m-%d")
schema_lines = "\n".join([f"- `{column}`: {dtype}" for column, dtype in final_df.dtypes.items()])

summary_markdown = f"""# Dataset Summary

## Data Source

Shopee Malaysia reviews from Google Play.

## Google Play Package Name

`{APP_PACKAGE}`

## Collection Method

Collected with `google-play-scraper` using `lang='en'`, `country='my'`, newest sorting, batch requests, and continuation tokens until the target of {TARGET_REVIEWS:,} reviews or the final available review page was reached.

## Collection Date

{collection_date}

## Review Counts

- Raw reviews collected: {total_raw_reviews:,}
- Cleaned reviews: {total_cleaned_reviews:,}
- Duplicates removed: {cleaning_stats['duplicates_removed']:,}
- Missing values removed: {cleaning_stats['null_removed']:,}

## NLP Preprocessing Workflow

1. Convert text to lowercase
2. Remove URLs
3. Remove email addresses
4. Remove HTML tags
5. Remove punctuation
6. Remove numbers
7. Remove emojis
8. Remove special characters
9. Remove extra whitespace
10. Tokenization
11. Remove English stopwords
12. Remove common Malay stopwords
13. English lemmatization

Important sentiment words such as `good`, `bad`, `excellent`, `terrible`, `love`, `hate`, `best`, and `worst` are kept.

## Final Dataset Schema

{schema_lines}
"""

SUMMARY_PATH.write_text(summary_markdown, encoding="utf-8")
print(f"Dataset documentation saved to: {SUMMARY_PATH}")
print(summary_markdown)

## 10. Kafka Sample Message

In [ ]:
sample_record = final_df.iloc[0].to_dict()
sample_record["rating"] = int(sample_record["rating"])
sample_record["thumbs_up_count"] = int(sample_record["thumbs_up_count"])

KAFKA_SAMPLE_PATH.write_text(
    json.dumps(sample_record, indent=4, ensure_ascii=False),
    encoding="utf-8",
)

print("Sample JSON record for Kafka Producer:")
print(json.dumps(sample_record, indent=4, ensure_ascii=False))
print(f"Kafka sample saved to: {KAFKA_SAMPLE_PATH}")

## 11. Final Verification

In [ ]:
required_files = [RAW_PATH, CLEAN_PATH, SUMMARY_PATH, KAFKA_SAMPLE_PATH, PROJECT_ROOT / "requirements.txt"]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Required output file is missing: {path}")
    print(f"FOUND: {path}")

expected_columns = [
    "review_id",
    "rating",
    "review_date",
    "app_version",
    "thumbs_up_count",
    "original_text",
    "cleaned_text",
]

if final_df.columns.tolist() != expected_columns:
    raise ValueError("Final dataset does not contain exactly the required columns.")

print("Final verification passed.")